# Image Enhancer CNN

Train a residual CNN on generated train/val/test splits. Input is RGB `[1, 3, 256, 256]`; output is `[brightness, contrast, saturation]`.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
NUM_WORKERS = 0
PARAMS = ["brightness", "contrast", "saturation"]
LEVELS = ["none", "small", "high"]
TRAIN_DISTORTION_LEVELS = ["none", "small", "high"]
STATS_SIZE = 18

ROOT = Path.cwd()
if ROOT.name != "ml" and (ROOT / "ml").exists():
    ROOT = ROOT / "ml"

DATA_DIR = ROOT / "data" / "processed"
LABELS_CSV = DATA_DIR / "labels.csv"
SPLIT_DIRS = {split: DATA_DIR / split for split in ("train", "val", "test")}
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"Train distortion levels: {TRAIN_DISTORTION_LEVELS}")

In [ ]:
def image_stats(image):
    flat = image.flatten(1)
    channel_mean = flat.mean(dim=1)
    channel_std = flat.std(dim=1, unbiased=False)
    channel_var = flat.var(dim=1, unbiased=False)
    channel_min = flat.min(dim=1).values
    channel_max = flat.max(dim=1).values
    global_flat = image.flatten()
    global_stats = torch.stack([
        global_flat.mean(),
        global_flat.std(unbiased=False),
        global_flat.var(unbiased=False),
    ])
    return torch.cat([channel_mean, channel_std, channel_var, channel_min, channel_max, global_stats]).float()


class EnhancementDataset(Dataset):
    def __init__(self, split, levels=None):
        self.split = split
        self.levels = set(levels or LEVELS)
        unknown = self.levels - set(LEVELS)
        if unknown:
            raise ValueError(f"Unknown distortion levels: {sorted(unknown)}")

        with LABELS_CSV.open("r", newline="", encoding="utf-8") as file:
            rows = list(csv.DictReader(file))

        self.rows = [
            row for row in rows
            if row["image_id"].startswith(f"{split}_") and row["distortion_level"] in self.levels
        ]
        if not self.rows:
            raise ValueError(f"No {split} samples for levels {sorted(self.levels)}")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        path = SPLIT_DIRS[self.split] / f"{row['image_id']}.jpg"
        image = Image.open(path).convert("HSV").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        image = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1)
        stats = image_stats(image)
        target = torch.tensor([float(row[name]) for name in PARAMS], dtype=torch.float32)
        level = torch.tensor(LEVELS.index(row["distortion_level"]), dtype=torch.long)
        return image, stats, target, level


def loader(split, levels=None, shuffle=False):
    dataset = EnhancementDataset(split, levels)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))


train_loader = loader("train", TRAIN_DISTORTION_LEVELS, shuffle=True)
val_loader = loader("val")
test_loader = loader("test")

print(f"Samples: train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")
image, stats, target, level = next(iter(train_loader))
image.shape, stats.shape, target.shape, level.shape

In [ ]:
class ResidualConvBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.layers(x))


class EnhancementCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))
        self.register_buffer("neutral", torch.tensor([[0.0, 1.0, 1.0]]))

        channels = [32, 64, 128, 192, 256]
        residual_dim = 32
        self.down_blocks = nn.ModuleList()
        self.to_residual = nn.ModuleList()
        self.from_residual = nn.ModuleList()
        in_channels = 3
        for level, out_channels in enumerate(channels):
            self.down_blocks.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                ResidualConvBlock(out_channels),
            ))
            self.to_residual.append(nn.Linear(out_channels, residual_dim))
            self.from_residual.append(nn.ModuleList([
                nn.Linear(residual_dim, out_channels) for _ in range(level)
            ]))
            in_channels = out_channels

        self.pool = nn.AdaptiveAvgPool2d(1)
        head_features = channels[-1] + STATS_SIZE + residual_dim * len(channels)
        self.head = nn.Sequential(
            nn.Linear(head_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.15),
            nn.Linear(128, 3),
        )

    def forward(self, image, stats):
        image = (image - self.mean) / self.std
        residuals = []
        features = image
        for level, block in enumerate(self.down_blocks):
            features = block(features)
            if residuals:
                additions = [project(residual).unsqueeze(-1).unsqueeze(-1) for project, residual in zip(self.from_residual[level], residuals)]
                features = features + torch.stack(additions, dim=0).sum(dim=0)
            pooled = self.pool(features).flatten(1)
            residuals.append(self.to_residual[level](pooled))
        features = torch.cat([self.pool(features).flatten(1), stats, *residuals], dim=1)
        return self.neutral + self.head(features)


model = EnhancementCNN().to(DEVICE)
model(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE), torch.zeros(1, STATS_SIZE, device=DEVICE)).shape

## Architecture diagram

```mermaid
flowchart LR
    image["HSV image<br/>B x 3 x 256 x 256"]
    stats["stats<br/>B x 18"]
    norm["normalize<br/>(image - 0.5) / 0.5"]

    image --> norm

    subgraph encoder["CNN encoder"]
        direction LR
        l1["Level 1<br/>Conv 3x3 stride 2: HSV 3 to 32<br/>BatchNorm + ReLU<br/>ResidualConvBlock 32<br/>B x 32 x 128 x 128"]
        l2["Level 2<br/>Conv 3x3 stride 2: 32 to 64<br/>BatchNorm + ReLU<br/>ResidualConvBlock 64<br/>add proj r1<br/>B x 64 x 64 x 64"]
        l3["Level 3<br/>Conv 3x3 stride 2: 64 to 128<br/>BatchNorm + ReLU<br/>ResidualConvBlock 128<br/>add proj r1 and r2<br/>B x 128 x 32 x 32"]
        l4["Level 4<br/>Conv 3x3 stride 2: 128 to 192<br/>BatchNorm + ReLU<br/>ResidualConvBlock 192<br/>add proj r1 r2 r3<br/>B x 192 x 16 x 16"]
        l5["Level 5<br/>Conv 3x3 stride 2: 192 to 256<br/>BatchNorm + ReLU<br/>ResidualConvBlock 256<br/>add proj r1 r2 r3 r4<br/>B x 256 x 8 x 8"]
    end

    norm --> l1 --> l2 --> l3 --> l4 --> l5

    p1["pool L1<br/>Linear 32 to 32<br/>r1"]
    p2["pool L2<br/>Linear 64 to 32<br/>r2"]
    p3["pool L3<br/>Linear 128 to 32<br/>r3"]
    p4["pool L4<br/>Linear 192 to 32<br/>r4"]
    p5["pool L5<br/>Linear 256 to 32<br/>r5"]

    l1 --> p1
    l2 --> p2
    l3 --> p3
    l4 --> p4
    l5 --> p5

    p1 -.-> l2
    p1 -.-> l3
    p1 -.-> l4
    p1 -.-> l5
    p2 -.-> l3
    p2 -.-> l4
    p2 -.-> l5
    p3 -.-> l4
    p3 -.-> l5
    p4 -.-> l5

    finalPool["final pool<br/>AdaptiveAvgPool2d 1<br/>flatten: B x 256"]
    concat["concat<br/>final pool B x 256<br/>stats B x 18<br/>r1..r5 B x 160<br/>total B x 434"]

    l5 --> finalPool --> concat
    stats --> concat
    p1 --> concat
    p2 --> concat
    p3 --> concat
    p4 --> concat
    p5 --> concat

    subgraph head["MLP head"]
        direction LR
        h1["Linear<br/>434 to 128"]
        h2["ReLU"]
        h3["Dropout<br/>p = 0.15"]
        h4["Linear<br/>128 to 3"]
    end

    concat --> h1 --> h2 --> h3 --> h4
    neutral["add neutral<br/>0.0 1.0 1.0"]
    output["params<br/>brightness contrast saturation"]
    h4 --> neutral --> output

    classDef inputNode fill:#d9e9ff,stroke:#3577c8,stroke-width:2px,color:#172033;
    classDef conv fill:#def4e7,stroke:#2f8f5b,stroke-width:2px,color:#172033;
    classDef residual fill:#fff0d8,stroke:#c87919,stroke-width:2px,color:#172033;
    classDef headNode fill:#eadfff,stroke:#7957c8,stroke-width:2px,color:#172033;
    classDef outputNode fill:#def4e7,stroke:#2f8f5b,stroke-width:2px,color:#172033;

    class image,stats inputNode;
    class l1,l2,l3,l4,l5 conv;
    class p1,p2,p3,p4,p5,neutral residual;
    class concat,h1,h2,h3,h4 headNode;
    class output outputNode;
```


In [ ]:
def run_epoch(model, data, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.SmoothL1Loss(reduction="none")
    total = 0.0
    total_mape = 0.0
    per_param = torch.zeros(3, device=DEVICE)
    per_param_mape = torch.zeros(3, device=DEVICE)
    per_level = torch.zeros(len(LEVELS), device=DEVICE)
    per_level_mape = torch.zeros(len(LEVELS), device=DEVICE)
    per_level_param = torch.zeros(len(LEVELS), 3, device=DEVICE)
    per_level_param_mape = torch.zeros(len(LEVELS), 3, device=DEVICE)
    level_count = torch.zeros(len(LEVELS), device=DEVICE)
    count = 0

    with torch.set_grad_enabled(training):
        for image, stats, target, level in data:
            image, stats, target, level = image.to(DEVICE), stats.to(DEVICE), target.to(DEVICE), level.to(DEVICE)
            pred = model(image, stats)
            param_loss = loss_fn(pred, target)
            param_mape = (pred.detach() - target).abs() / target.abs().clamp_min(1e-3) * 100.0
            sample_mape = param_mape.mean(dim=1)
            sample_loss = param_loss.mean(dim=1)
            loss = sample_loss.mean()

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            total += sample_loss.detach().sum().item()
            total_mape += sample_mape.detach().sum().item()
            per_param += param_loss.detach().sum(dim=0)
            per_param_mape += param_mape.sum(dim=0)
            count += image.size(0)
            for level_index in range(len(LEVELS)):
                mask = level == level_index
                if mask.any():
                    per_level[level_index] += sample_loss.detach()[mask].sum()
                    per_level_mape[level_index] += sample_mape.detach()[mask].sum()
                    per_level_param[level_index] += param_loss.detach()[mask].sum(dim=0)
                    per_level_param_mape[level_index] += param_mape[mask].sum(dim=0)
                    level_count[level_index] += mask.sum()

    metrics = {
        "total": total / count,
        "param": (per_param / count).cpu().tolist(),
        "total_mape": total_mape / count,
        "mape": (per_param_mape / count).cpu().tolist(),
        "level": {},
    }
    for i, name in enumerate(LEVELS):
        if level_count[i] > 0:
            metrics["level"][name] = {
                "total": (per_level[i] / level_count[i]).item(),
                "total_mape": (per_level_mape[i] / level_count[i]).item(),
                "total_mape": (per_level_mape[i] / level_count[i]).item(),
                "param": (per_level_param[i] / level_count[i]).cpu().tolist(),
                "mape": (per_level_param_mape[i] / level_count[i]).cpu().tolist(),
            }
    return metrics


def metric_table(title, metrics_by_split, include_mape=False):
    rows = [f"\n{title}", "split    level      total   brightness   contrast   saturation"]
    rows.append("-" * len(rows[-1]))
    for split, metrics in metrics_by_split.items():
        p = metrics["param"]
        rows.append(f"{split:<8} {'all':<7} {metrics['total']:>8.5f} {p[0]:>12.5f} {p[1]:>10.5f} {p[2]:>12.5f}")
        for level in LEVELS:
            values = metrics["level"].get(level)
            if values is None:
                continue
            lp = values["param"]
            rows.append(f"{split:<8} {level:<7} {values['total']:>8.5f} {lp[0]:>12.5f} {lp[1]:>10.5f} {lp[2]:>12.5f}")

    if not include_mape:
        return "\n".join(rows)

    rows += ["", "MAPE (%)", "split    level      total   brightness   contrast   saturation"]
    rows.append("-" * len(rows[-1]))
    for split, metrics in metrics_by_split.items():
        p = metrics["mape"]
        rows.append(f"{split:<8} {'all':<7} {metrics['total_mape']:>8.2f} {p[0]:>12.2f} {p[1]:>10.2f} {p[2]:>12.2f}")
        for level in LEVELS:
            values = metrics["level"].get(level)
            if values is None:
                continue
            lp = values["mape"]
            rows.append(f"{split:<8} {level:<7} {values['total_mape']:>8.2f} {lp[0]:>12.2f} {lp[1]:>10.2f} {lp[2]:>12.2f}")
    return "\n".join(rows)


def train_model(model):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    best_state, best_val = None, float("inf")
    for epoch in tqdm(range(EPOCHS), desc="Training"):
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)
        if val_metrics["total"] < best_val:
            best_val = val_metrics["total"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        tqdm.write(metric_table(f"Epoch {epoch + 1:02d}", {"train": train_metrics, "val": val_metrics}))

    model.load_state_dict(best_state)
    return model


model = train_model(model)
print(metric_table("Test", {"test": run_epoch(model, test_loader)}, include_mape=True))

In [ ]:
def export_onnx(model, path):
    model.eval().cpu()
    dummy_image = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
    dummy_stats = torch.zeros(1, STATS_SIZE, dtype=torch.float32)
    torch.onnx.export(
        model,
        (dummy_image, dummy_stats),
        path,
        input_names=["image", "stats"],
        output_names=["params"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )
    return path


checkpoint_path = MODEL_DIR / "enhancer.onnx"
export_onnx(model, checkpoint_path)
checkpoint_path